In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [18]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_community.vectorstores import InMemoryVectorStore 
from langchain.tools import tool
from langchain.agents import create_agent

In [4]:
loader=PyPDFLoader(
    "../data/medical_report.pdf",
)
docs=loader.load()


In [5]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(docs)

In [7]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")
vector_store=InMemoryVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings
)

# agent=tools, llm,prompt


In [ ]:
@tool
def retrival_tool(query:str):
    """
        this tool can help you to retrieve relevant data of the pdf documents,
        and these pdf documents have details about medical reports
    """
    same_record=vector_store.similarity_search(query=query,k=4)
    # return same_record
    context=""
    for doc in same_record:
        context+=doc.page_content+"\n\n"
    return context

In [17]:
llm=ChatGroq(model="openai/gpt-oss-20b")

In [19]:
from langchain.agents import create_agent
system_prompt="""
    you are a helpul assistant that answers using retrival tool 
    Always use the 'retrieval_tool' tool for question requiring external knowledge
"""
agent=create_agent(
    model=llm,
    tools=[retrival_tool],
    system_prompt=system_prompt
)
  

In [20]:
query="what is the name of patient and what is the name of the doctors"
res=agent.invoke({"messages":[{"role":"user","content":query}]})

In [21]:
print(res["messages"][-1].content)

**Patient name**  
- Ms. Nikita Chudhary  

**Doctor(s) listed on the report**  
- Dr Nitin Nahar (referring physician)  
- Dr Sunanda, MD, Pathology (Senior Consultant Pathologist – Hematology & Immunology)  

These details are taken from the laboratory report you provided.
